In [0]:
%sql
CREATE WIDGET TEXT prm_start_year_month DEFAULT "201001";
CREATE WIDGET TEXT prm_end_year_month DEFAULT "202602";

In [0]:
start_year_month = dbutils.widgets.get("prm_start_year_month")
end_year_month = dbutils.widgets.get("prm_end_year_month")

In [0]:
%sql
USE CATALOG nyc; 
USE SCHEMA process_silver;

In [0]:
from pyspark.sql import functions as F 

base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"
delta_target = "nyc.process_silver.delta_yellow_taxi"

In [0]:
import datetime
from dateutil.relativedelta import relativedelta 

def get_file_paths_list(start_year_month, end_year_month):
    start = datetime.datetime.strptime(start_year_month, "%Y%m")
    end = datetime.datetime.strptime(end_year_month, "%Y%m")
    current = start 
    months = []
    while current <= end: 
        months.append(current.strftime("%Y%m")) 
        current += relativedelta(months=1)
    return months

def load_data_to_bronze(start_time, end_time, base_path):
    # 1. Generate the list of months
    target_months = get_file_paths_list(start_time, end_time)
    
    # 2. change format from YYYYMM to YYYY-MM
    to_format = lambda m: datetime.datetime.strptime(m, '%Y%m').strftime('%Y-%m')
    
    # 3. Generate the full path
    input_paths = [f"{base_path.rstrip('/')}/yellow_tripdata_{to_format(m)}.parquet" for m in target_months]
    
    # Get the actual file paths on disk
    raw_files = dbutils.fs.ls(base_path)
    
    # save the paths to a set to improve the performance
    actual_files_set = {f.path.replace("dbfs:", "") for f in raw_files}
    
    # Filter the files in disk between start year month number and end year month number 
    valid_paths = [p for p in input_paths if p.replace("dbfs:", "") in actual_files_set]
    
    return valid_paths

In [0]:
months = load_data_to_bronze(start_year_month, end_year_month, yellow_path)


In [0]:
months